# 08. Workflow с retry и идемпотентностью

Мини-движок пакетной обработки на чистом Python.

Стек: Python standard library. Выполняйте ячейки сверху вниз.

In [ ]:
import time
processed=set()

def handle(item):
    key=item['id']
    if key in processed: return 'duplicate-skipped'
    if item.get('temporary_error'): raise ConnectionError('temporary')
    processed.add(key); return 'done'

def run(items,max_attempts=2):
    results=[]
    for item in items:
        for attempt in range(max_attempts):
            try:
                results.append((item['id'],handle(item))); break
            except ConnectionError:
                if attempt+1==max_attempts: results.append((item['id'],'manual-review'))
                else: time.sleep(.1)
    return results

run([{'id':'a'},{'id':'b','temporary_error':True},{'id':'a'}])